In [27]:
!pip install pyspark

In [28]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Expense_monitoring").getOrCreate()

In [29]:
from google.colab import files

uploaded = files.upload()

Saving expenses_cleaned.csv to expenses_cleaned (2).csv


In [42]:
expenses_df = spark.read.csv(
    "expenses_cleaned.csv",
    header=True,
    inferSchema=True
    )

In [43]:
expenses_df.printSchema()

root
 |-- expense_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- expense_date: date (nullable = true)
 |-- description: string (nullable = true)
 |-- month: timestamp (nullable = true)



In [44]:
expenses_df.show(5)

+----------+-------+---------+---------+------+------------+-------------------+-------------------+
|expense_id|user_id|user_name| category|amount|expense_date|        description|              month|
+----------+-------+---------+---------+------+------------+-------------------+-------------------+
|         1|      1|     Arun|Transport|246.73|  2026-01-01|Metro card recharge|2026-01-01 00:00:00|
|         2|      2|    Priya|     Food|360.05|  2026-01-01|             Snacks|2026-01-01 00:00:00|
|         3|      2|    Priya|     Food|120.66|  2026-01-01|          Groceries|2026-01-01 00:00:00|
|         4|      2|    Priya|     Food|383.96|  2026-01-01|              Lunch|2026-01-01 00:00:00|
|         5|      3|    Kavin|Transport|198.93|  2026-01-01|            Unknown|2026-01-01 00:00:00|
+----------+-------+---------+---------+------+------------+-------------------+-------------------+
only showing top 5 rows


In [45]:
from pyspark.sql.types import DoubleType

#check amount is in double type
expenses_df = expenses_df.withColumn("amount", expenses_df["amount"].cast(DoubleType()))

In [46]:
#change the column name from month to transaction_date
from pyspark.sql import functions as F
expenses_df = expenses_df.withColumn(
    'transaction_date',
    F.col('month').cast('date')
).withColumn(
    'month',
    F.month('transaction_date')
)

#Total monthly spend per user
group_by_month_df = expenses_df.groupBy('user_id','month').agg(
    F.round(F.sum('amount'),2).alias('total_monthly_spend')
).orderBy("user_id", "month")
group_by_month_df.show()


+-------+-----+-------------------+
|user_id|month|total_monthly_spend|
+-------+-----+-------------------+
|      1|    1|           21251.53|
|      1|    2|           47717.05|
|      1|    3|           21743.12|
|      1|    4|           20206.63|
|      1|    5|           42919.42|
|      1|    6|           25037.48|
|      2|    1|           29597.22|
|      2|    2|           31801.69|
|      2|    3|            26748.4|
|      2|    4|           47343.02|
|      2|    5|           26473.98|
|      2|    6|           22266.72|
|      3|    1|           30572.74|
|      3|    2|           27091.38|
|      3|    3|           44666.33|
|      3|    4|           29635.98|
|      3|    5|           26001.79|
|      3|    6|           42977.87|
|      4|    1|           50634.76|
|      4|    2|           23620.97|
+-------+-----+-------------------+
only showing top 20 rows


In [50]:
# A. Large one-time expenses: on category
from pyspark.sql.window import Window
w_cat = Window.partitionBy("category")
large_df = expenses_df.withColumn(
    "category_avg",
    F.round(F.avg("amount").over(w_cat), 2)
).filter(F.col("amount") > 3 * F.col("category_avg")).orderBy(F.desc("amount"))

# Monthly spikes
w_user = Window.partitionBy("user_id")
spike_df = (group_by_month_df
    .withColumn("user_median", F.percentile_approx("total_monthly_spend", 0.5).over(w_user))
    .filter(F.col("total_monthly_spend") > 1.3 * F.col("user_median")))
large_df.show()
spike_df.show()

+----------+-------+---------+-------------+--------+------------+--------------------+-----+----------------+------------+
|expense_id|user_id|user_name|     category|  amount|expense_date|         description|month|transaction_date|category_avg|
+----------+-------+---------+-------------+--------+------------+--------------------+-----+----------------+------------+
|       445|      1|     Arun|     Shopping|29390.69|  2026-02-20|     Laptop purchase|    2|      2026-02-01|     2618.35|
|       924|      2|    Priya|    Education|25140.28|  2026-04-18|Certification pro...|    4|      2026-04-01|     3195.71|
|       452|      8|   Anitha|Entertainment|24846.73|  2026-02-20|             Unknown|    2|      2026-02-01|     1282.46|
|       107|      4|    Meena|Entertainment|24696.41|  2026-01-13|    Destination trip|    1|      2026-01-01|     1282.46|
|       691|      3|    Kavin|     Shopping|24614.19|  2026-03-21|          Smartphone|    3|      2026-03-01|     2618.35|
|      1